# REVE v4-multi: обучение на двух больницах (NMT + Турция), проверка на третьей (EpiEEG-60)

| Поле | Значение |
|------|----------|
| **Автор** | Захар Санников (zs) |
| **Дата** | 2026-10-29 |
| **Гипотеза** | Г1 (ЭЭГ) |
| **Вопрос** | REVE v4 обучена только на NMT: NMT eval 0,917, Турция 0,915, EpiEEG-60 0,761. Улучшит ли перенос в третью больницу обучение сразу на двух больницах? В обоих источниках есть оба класса, так что источник класс не подсказывает |
| **Ядра Kaggle** | `zakharsann/h1-reve-multisite` (GPU T4), короткая проверка `zakharsann/h1-reve-multisite-smoke` (процессор) |
| **Источники** | `zakharsann/h1-prep-shared` (NMT + Турция), `zakharsann/h1-reve-v4` (код модели и оценки v4 по людям), `zakharsann/h1-epieeg60-prep` (сигнал EpiEEG-60), `zakharsann/h1-epieeg60-reve` (оценки v4 на EpiEEG-60, пометки разрядов) |
| **Деление Турции** | 5 фолдов по людям, стратифицировано по классу (StratifiedKFold, seed 0). Фолд k: обучение на NMT fit + 4/5 Турции, проверка на оставшейся 1/5. Из обучающих 4/5 Турции 20 % людей (стратифицировано, seed k) отложены для выбора эпохи и порога. Оценка на всей Турции — out-of-fold (каждый человек оценён моделью, которая его не видела) |
| **Запуски** | 5 фолдов × 1 запуск (seed = номер фолда). 5 × 3 запусков (~4 ч) в бюджет 2,5 ч не влезает. Для NMT eval и EpiEEG-60 — ансамбль 5 моделей фолдов (среднее вероятностей); для честного сравнения с ансамблем 3 запусков v4 — ещё ансамбль фолдов 0–2 |
| **Обучение** | как v4: `brain-bzh/reve-base` + голова, окна 10 с, новое случайное окно на каждой эпохе, нормировка окна, аугментации (шум 0,1·СКО, зануление 1–2 каналов, замена полушарий, сдвиг до 2 с), основа 1·10⁻⁵, голова 5·10⁻⁴, AdamW 0,05, пакет 32, fp16, до 10 эпох, косинус с разогревом 5 %, остановка через 2 эпохи без роста. За эпоху столько же окон, сколько у v4 (8 × 1894 = 15 152) |
| **Баланс** | записи выбираются с весами: NMT — 70 % окон эпохи, Турция — 30 %; внутри каждого источника классы поровну. Без весов Турция (~77 записей против 1894) дала бы ~4 % окон |
| **Выбор эпохи** | среднее двух ROC AUC: отложенные 15 % NMT train (335 записей) и отложенные 20 % турецкого обучения (~19–20 человек); 12 окон на запись равномерно, медиана (как у v4) |
| **Оценка человека** | 60 окон равномерно, среднее верхних 20 % окон (основной вид v4); дополнительно — медиана |
| **Порог** | Юден и чувствительность 80 % по отложенной части обучения. Турция (out-of-fold): порог каждого фолда — по его отложенным (NMT val + турецкие отложенные, оценки модели фолда). NMT eval и EpiEEG-60: порог ансамбля — по NMT val (ансамбль) + турецким людям, бывшим отложенными (оценка модели фолда, где человек был отложен) |
| **Метрики** | ROC AUC, accuracy, сбалансированная accuracy, precision, recall, специфичность — 95% ДИ бутстреп по людям (2000); парный бутстреп «multi − v4»; калибровка порога на k = 10/20 людях EpiEEG-60 (500 повторов); ROC AUC у больных EpiEEG-60 с размеченными разрядами и без |

**Веса в выходе ядра** (в git не кладутся): `reve_multi_fold{0..4}.pt` — `{"state_dict", "config"}`, класс `RVNet` и `rv_input` — в `reve_multi_model.py` (тот же код, что `reve_v4_model.py`). В `config` — номер фолда и id турецких людей: обучение / отложенные / проверка.

## Выводы

**Гипотеза «обучение на двух больницах улучшит перенос в третью» не подтвердилась.** На EpiEEG-60 REVE v4-multi (ансамбль 5 моделей фолдов) не лучше v4, а по точечной оценке чуть хуже.

1. **EpiEEG-60 (n = 63, 31 эпилепсия), 60 окон, верхние 20 %:** ROC AUC multi **0,712 (0,58–0,84)** против v4 **0,761 (0,63–0,88)**. Парный бутстреп «multi − v4»: **−0,049 (−0,128…+0,027)**, интервал включает 0. Ансамбль фолдов 0–2 (равный по размеру v4): 0,718, разность −0,043 (−0,115…+0,026). Отдельные модели фолдов: 0,70 / 0,70 / 0,70 / 0,67 / 0,72. Медиана окон: 0,707 против 0,748, разность −0,041 (−0,102…+0,021). Оценки multi и v4 тесно связаны (корреляция 0,92): вторая больница почти не изменила, что модель «видит» в третьей.
2. **NMT eval (n = 185):** 0,907 (0,87–0,94) против 0,913; разность −0,006 (−0,028…+0,014) — без потерь от добавления Турции.
3. **Турция, out-of-fold (n = 121):** 0,957 (0,92–0,99), по фолдам 0,88 / 1,00 / 0,99 / 0,96 / 0,97; v4 (Турцию не видела) — 0,923. Разность +0,034 (−0,012…+0,083). Условия неравные: multi учили на других турецких людях, поэтому это не перенос, а внутрибольничная оценка. Рост небольшой и не значим.
4. **Пороговые метрики EpiEEG-60** при пороге с отложенной части обучения (Юден 0,886): accuracy 0,63 (0,52–0,75), сбаланс. accuracy 0,63, precision 0,70, recall **0,45 (0,29–0,63)**, специфичность 0,81. У v4 при пороге 0,832: recall 0,61, сбаланс. accuracy 0,70. Порог с чувствительностью 80 % по отложенным (0,864) даёт на EpiEEG-60 recall 0,61, специфичность 0,81.
5. **Калибровка на месте** (500 повторов, порог по Юдену): сбаланс. accuracy у multi 0,64 (0,51–0,74) при k = 10 и 0,67 (0,52–0,76) при k = 20; у v4 — 0,68 и 0,69. Калибровка не помогает ни той, ни другой.
6. **Подгруппы EpiEEG-60:** больные с пометками разрядов (24) против всех здоровых: multi 0,741 (0,60–0,88), v4 0,807; без пометок (7): multi 0,612 (0,36–0,84), v4 0,603, разность +0,009 (−0,121…+0,130). Случаи без явных разрядов остаются трудными (recall при пороге 0,14); интервалы очень широкие (7 человек).
7. **Ограничения.** Один запуск на фолд (5 × 3 запусков не влезло в бюджет); модели фолдов — одиночные, ансамбль 5 сравнивается с ансамблем 3 запусков v4, поэтому дополнительно дан ансамбль фолдов 0–2. На турецких отложенных (20 человек) AUC выбора эпохи почти 1,0 — выбор эпохи фактически идёт по NMT, и порог по 20 человекам шумный. Различия multi и v4 на EpiEEG-60 в пределах шума (ДИ включает 0), но и улучшения нет.

## Отличия от задания и выбор
- **5 фолдов × 1 запуск** (≈3,6 мин/эпоху, 4–7 эпох на фолд, всего 112 мин обучения; 5 × 3 не влезло в 2,5 ч).
- **Баланс:** 70 % окон эпохи — NMT, 30 % — Турция (иначе ≈4 %); внутри источника классы поровну. Турция в обучении фолда попадает в выборку примерно в 10 раз чаще на запись, чем NMT — риск запоминания, поэтому отложенные по людям.
- **Модель для EpiEEG-60:** ансамбль 5 моделей фолдов (вместе они видели всю Турцию; одна модель на всей Турции не обучалась).
- **Порог ансамбля** (NMT eval, EpiEEG-60): Юден/80 % по NMT val + турецким людям, оценённым моделью фолда, где они были отложенными (по 20 человек в каждом фолде). Для Турции out-of-fold — свой порог у каждого фолда.

## Результаты и файлы
`h1_eeg/results/reve_multi_{summary,paired_differences,threshold_metrics,epieeg60_calibration,epieeg60_subgroups,per_person,runs,turkey_folds}.csv`, `reve_multi_window_scores.csv.gz`; картинки `results/figures/reve_multi_roc.png`, `reve_multi_epieeg60_scores.png`. Веса — в выходе ядра `zakharsann/h1-reve-multisite` (`reve_multi_fold{0..4}.pt`, по 287 МБ, в git не кладутся; в `config` каждого — id турецких людей: обучение / отложенные / проверка).



In [ ]:
# служебная пометка начала ячейки — по ней вывод из лога Kaggle раскладывается по ячейкам
print("@@cell 1")
"""Общий набор Г1 — копия h1_eeg/src/shared_data.py."""
import glob, json, os
from fractions import Fraction
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, scipy.signal as ss

EL = ["Fp1", "Fp2", "F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2", "F7", "F8", "T3", "T4", "T5", "T6", "Fz", "Cz", "Pz"]


def find_root(base="/kaggle/input"):
    """Папка общего набора среди подключённых источников."""
    hits = glob.glob(f"{base}/**/h1_shared_meta.csv", recursive=True)
    assert hits, "общий набор не подключён: добавьте zakharsann/h1-prep-shared в kernel_sources"
    return os.path.dirname(hits[0])


def load_meta(root):
    """Таблица записей и описание набора."""
    meta = pd.read_csv(f"{root}/h1_shared_meta.csv")
    info = json.load(open(f"{root}/h1_shared_info.json", encoding="utf-8"))
    return meta, info


_SHARDS = {}
def _shard(root, fs, name):
    k = (root, fs, name)
    if k not in _SHARDS:
        _SHARDS[k] = np.load(f"{root}/fs{fs}/{name}", mmap_mode="r")
    return _SHARDS[k]


def read_record(root, row, fs=200):
    """Одна запись: (19, n) float16, мкВ. row — строка meta (Series или dict).
    Через mmap — для единичных записей; много записей читать через read_all (на /kaggle/input mmap очень медленный)."""
    n, off = int(row[f"n{fs}"]), int(row[f"off{fs}"])
    return np.array(_shard(root, fs, row[f"shard{fs}"])[off:off + 19 * n]).reshape(19, n)


def read_all(root, meta, fs=200, threads=4):
    """Все записи meta в словарь id -> (19, n) float16: каждый файл-кусок читается целиком последовательно, записи режутся в памяти."""
    def load(item):
        name, g = item
        arr = np.load(f"{root}/fs{fs}/{name}")
        return [(i, arr[o:o + 19 * n].reshape(19, n).copy()) for i, o, n in zip(g["id"], g[f"off{fs}"], g[f"n{fs}"])]
    out = {}
    with ThreadPoolExecutor(threads) as ex:
        for part in ex.map(load, list(meta.groupby(f"shard{fs}"))):
            out.update(part)
    return {i: out[i] for i in meta["id"]}


def windows(X, L, S=None):
    """Окна длиной L с шагом S (по умолчанию S = L, подряд): (k, 19, L)."""
    S = S or L
    k = (X.shape[-1] - L) // S + 1 if X.shape[-1] >= L else 0
    if k <= 0:
        return np.zeros((0,) + X.shape[:-1] + (L,), X.dtype)
    return np.lib.stride_tricks.sliding_window_view(X, L, axis=-1)[..., ::S, :][..., :k, :].swapaxes(0, -2).copy()


def random_windows(X, L, K, rng):
    """K окон длиной L со случайным началом (любой отсчёт) — свежие окна на каждой эпохе: (K, 19, L)."""
    n = X.shape[-1] - L + 1
    if n <= 0:
        return np.zeros((0,) + X.shape[:-1] + (L,), X.dtype)
    st = np.sort(rng.integers(0, n, K))
    return np.stack([X[..., s:s + L] for s in st])


def resample(X, fs_from, fs_to):
    """Пересчёт частоты (полифазный, как scipy.signal.resample_poly) по последней оси, float32 на выходе."""
    if fs_from == fs_to:
        return X
    fr = Fraction(int(fs_to), int(fs_from))
    return ss.resample_poly(np.asarray(X, np.float32), fr.numerator, fr.denominator, axis=-1).astype(np.float32)


def highpass(X, f, fs=200, order=4):
    """Дофильтровка сверху общей нижней границы 0,1 Гц (например, 0,3 Гц как у CBraMod): по всей записи, не по окну."""
    sos = ss.butter(order, f, btype="highpass", fs=fs, output="sos")
    return ss.sosfiltfilt(sos, np.asarray(X, np.float64), axis=-1).astype(np.float32)

In [ ]:
import os, glob, time, json, io
SMOKE = False   # True — короткая проверка на процессоре Kaggle (часть записей, 2 фолда, 1 эпоха); её метрики не смотреть
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, train_test_split
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from PIL import Image
DEV = "cuda" if torch.cuda.is_available() else "cpu"
OUT = "/kaggle/working"; FIG = f"{OUT}/figures"; os.makedirs(FIG, exist_ok=True)
T_NB = time.time()
END = T_NB + (45 if SMOKE else 138) * 60      # весь блокнот — не дольше 2 ч 18 мин T4 (бюджет 2,5 ч)
left = lambda: (END - time.time()) / 60

SFREQ, SEG = 200, 10
L = SFREQ * SEG
K_TRAIN, N_VAL_EP, CAP, HP = 8, (2 if SMOKE else 12), (4 if SMOKE else 60), 0.3
N_FOLDS, SHARE_TK, IV_FRAC = (2 if SMOKE else 5), 0.3, 0.2

V4 = os.path.dirname(sorted(glob.glob("/kaggle/input/**/reve_v4_model.py", recursive=True))[0])
EPI = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/epieeg60_records.csv", recursive=True) if os.path.isdir(os.path.join(os.path.dirname(p), "fs200"))][0]
V4_PP = sorted(glob.glob("/kaggle/input/**/reve_v4_per_person.csv", recursive=True))[0]
V4_EPI = sorted(glob.glob("/kaggle/input/**/epieeg60_per_person.csv", recursive=True))[0]
print("источники:", V4, EPI, V4_PP, V4_EPI, sep="\n  ")

# модель и подготовка окна — код REVE v4 без изменений; копия в выход как reve_multi_model.py
MODEL_SRC = open(f"{V4}/reve_v4_model.py", encoding="utf-8").read()
open(f"{OUT}/reve_multi_model.py", "w", encoding="utf-8").write(MODEL_SRC)
exec(compile(MODEL_SRC, "reve_multi_model.py", "exec"))
POS = positions(DEV)

MIRROR_PAIRS = {"Fp1": "Fp2", "F3": "F4", "C3": "C4", "P3": "P4", "O1": "O2", "F7": "F8", "T3": "T4", "T5": "T6"}
MIRROR_PAIRS.update({v: k for k, v in list(MIRROR_PAIRS.items())})
MIRROR = [EL.index(MIRROR_PAIRS.get(e, e)) for e in EL]
AMP = dict(device_type="cuda", dtype=torch.float16, enabled=DEV == "cuda")

def predict_windows(model, Ws, bs=64):
    """Вероятность каждого окна: список массивов, по одному на запись."""
    model.eval(); out = []
    with torch.no_grad(), torch.autocast(**AMP):
        for W in Ws:
            p = [torch.sigmoid(model(rv_input(W[b:b + bs]).to(DEV)).float()).cpu().numpy() for b in range(0, len(W), bs)]
            out.append(np.concatenate(p) if p else np.zeros(0, np.float32))
    return out

def spread(n, k):
    return np.unique(np.linspace(0, n - 1, k).astype(int)) if n > k else np.arange(n)
print("устройство:", torch.cuda.get_device_name(0) if DEV == "cuda" else "процессор", "| ядер процессора:", os.cpu_count(), "| позиции:", tuple(POS.shape))

источники:
  /kaggle/input/notebooks/zakharsann/h1-reve-v4
  /kaggle/input/notebooks/zakharsann/h1-epieeg60-prep
  /kaggle/input/notebooks/zakharsann/h1-reve-v4/reve_v4_per_person.csv
  /kaggle/input/notebooks/zakharsann/h1-epieeg60-reve/epieeg60_per_person.csv
устройство: Tesla T4 | ядер процессора: 4 | позиции: (19, 3)


## 1. Данные: NMT, Турция (5 фолдов по людям), EpiEEG-60

In [ ]:
ROOT = find_root()
meta, info = load_meta(ROOT)
need = meta[meta.split.isin(["fit", "val", "eval", "turkey"])].copy()
if SMOKE:
    pick = lambda s, n0, n1: pd.concat([need[(need.split == s) & (need.y == 0)].head(n0), need[(need.split == s) & (need.y == 1)].head(n1)])
    need = pd.concat([pick("fit", 16, 8), pick("val", 5, 5), pick("eval", 4, 4), pick("turkey", 10, 10)])
t0 = time.time()
REC = read_all(ROOT, need)
t_read = time.time() - t0
def hp16(i):
    """ФВЧ 0,3 Гц по записи целиком (как у REVE v4), хранение снова float16."""
    return i, highpass(REC[i], HP).astype(np.float16)
t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    for i, X in ex.map(hp16, list(REC)):
        REC[i] = X
print(f"прочитано {len(REC)} записей за {t_read:.0f} с, ФВЧ 0,3 Гц {time.time()-t0:.0f} с | {sum(x.nbytes for x in REC.values())/1e9:.1f} ГБ | осталось {left():.0f} мин")

fit, val, ev, tk = (need[need.split == s].reset_index(drop=True) for s in ["fit", "val", "eval", "turkey"])
print(pd.crosstab(need.split, need.y))
if not SMOKE:
    assert (len(fit), len(val), len(ev), len(tk)) == (1894, 335, 185, 121), (len(fit), len(val), len(ev), len(tk))
assert tk.id.is_unique      # в турецком наборе одна запись (.mat) на человека

# Турция: 5 фолдов по людям, стратифицировано; внутри обучающих 4/5 — 20 % отложенных (по людям, стратифицировано)
tk["fold"] = -1
for k, (_, te) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=0).split(tk.id, tk.y)):
    tk.loc[te, "fold"] = k
ROLE = {}          # фолд -> Series id -> fit / iv / test
for k in range(N_FOLDS):
    tr = tk[tk.fold != k]
    _, iv = train_test_split(tr, test_size=IV_FRAC, stratify=tr.y, random_state=k)
    ROLE[k] = pd.Series(np.where(tk.fold == k, "test", np.where(tk.id.isin(iv.id), "iv", "fit")), index=tk.id)
TKF = pd.DataFrame({"id": tk.id, "y": tk.y, "fold": tk.fold, **{f"role_f{k}": ROLE[k].values for k in ROLE}})
print(TKF.groupby("fold").y.agg(["size", "sum"]).rename(columns={"size": "людей", "sum": "эпилепсия"}).T.to_string())
print("по фолдам (обучение / отложенные / проверка):", {k: ROLE[k].value_counts().reindex(["fit", "iv", "test"]).tolist() for k in ROLE})

# EpiEEG-60: подготовка части 1 + ФВЧ 0,3 Гц; окна всех записей человека подряд, 60 равномерно
ER = pd.read_csv(f"{EPI}/epieeg60_records.csv")
ER = ER[ER.n200.notna()].sort_values(["person", "id"]).reset_index(drop=True)
if SMOKE:
    ER = pd.concat([ER[ER.y == 0].head(3), ER[ER.y == 1].head(3)]).reset_index(drop=True)
def load_epi(i):
    X = highpass(np.load(f"{EPI}/fs200/{i}.npy"), HP)
    return i, windows(X.astype(np.float16), L)
with ThreadPoolExecutor(4) as ex:
    EW = dict(ex.map(load_epi, ER.id))
epi = ER.groupby("person", sort=True).agg(y=("y", "first")).reset_index()
EPIW = {}
for p, g in ER.groupby("person", sort=True):
    W = np.concatenate([EW[i] for i in g.id]); EPIW[p] = W[spread(len(W), CAP)]
del EW
print(f"EpiEEG-60: {len(epi)} людей (эпилепсия {epi.y.sum()}), записей {len(ER)} | осталось {left():.0f} мин")

def w60(i):
    W = windows(REC[i], L); return W[spread(len(W), CAP)]
GET = {"nmt_val": (val.id.tolist(), w60), "nmt_eval": (ev.id.tolist(), w60), "turkey": (tk.id.tolist(), w60), "epieeg60": (epi.person.tolist(), EPIW.__getitem__)}
YS = {"nmt_val": val.y.to_numpy(), "nmt_eval": ev.y.to_numpy(), "turkey": tk.y.to_numpy(), "epieeg60": epi.y.to_numpy()}
def predict_set(model, ds):
    ids, f = GET[ds]; return predict_windows(model, (f(i) for i in ids))

# окна для выбора эпохи: 12 на запись равномерно (как у v4)
y_val = val.y.to_numpy()
VAL12 = [windows(REC[i], L)[spread(REC[i].shape[1] // L, N_VAL_EP)] for i in val.id]
TK12 = {i: windows(REC[i], L)[spread(REC[i].shape[1] // L, N_VAL_EP)] for i in tk.id}

прочитано 2535 записей за 40 с, ФВЧ 0,3 Гц 92 с | 14.6 ГБ | осталось 135 мин
y          0    1
split            
eval      95   90
fit     1618  276
turkey    71   50
val      286   49
fold        0   1   2   3   4
людей      25  24  24  24  24
эпилепсия  10  10  10  10  10
по фолдам (обучение / отложенные / проверка): {0: [76, 20, 25], 1: [77, 20, 24], 2: [77, 20, 24], 3: [77, 20, 24], 4: [77, 20, 24]}
EpiEEG-60: 63 людей (эпилепсия 31), записей 63 | осталось 135 мин


## 2. Обучение: 5 фолдов, NMT fit + 4/5 Турции, веса 70 / 30 по источникам

In [ ]:
NOISE, BS, MAX_EP, PATIENCE, LR_HEAD, LR_BACK = 0.1, (8 if SMOKE else 32), (1 if SMOKE else 10), 2, 5e-4, 1e-5
def augment(x, g):
    B = x.shape[0]
    x = x + NOISE * torch.randn_like(x)
    m = torch.rand(B, device=x.device) < 0.5
    x[m] = x[m][:, MIRROR]
    for b in range(B):
        if g.random() < 0.5:
            x[b, g.choice(19, g.integers(1, 3), replace=False)] = 0
        x[b] = torch.roll(x[b], int(g.integers(-2 * SFREQ, 2 * SFREQ + 1)), dims=-1)
    return x

def make_opt(model, total_steps):
    back = [p for n, p in model.named_parameters() if n.startswith("backbone")]
    head = [p for n, p in model.named_parameters() if not n.startswith("backbone")]
    opt = torch.optim.AdamW([{"params": back, "lr": LR_BACK}, {"params": head, "lr": LR_HEAD}], weight_decay=0.05)
    warm_n = max(1, int(0.05 * total_steps))
    f = lambda s: (s + 1) / warm_n if s < warm_n else 0.5 * (1 + np.cos(np.pi * (s - warm_n) / max(1, total_steps - warm_n))) * 0.99 + 0.01
    return opt, torch.optim.lr_scheduler.LambdaLR(opt, f)

def step(model, opt, sched, scaler, x, t, params):
    with torch.autocast(**AMP):
        out = model(x)
    loss = F.binary_cross_entropy_with_logits(out.float(), t.float())
    bad = not torch.isfinite(loss)
    opt.zero_grad()
    if not bad:
        scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update()
    sched.step()
    return loss.item(), bad

N_EP = 16 if SMOKE else K_TRAIN * len(fit)          # окон за эпоху — столько же, сколько у v4 (15 152)
def train_fold(k, deadline):
    """Фолд k: NMT fit + турецкие люди с ролью fit. Каждое окно — новое случайное. Выбор эпохи — среднее ROC AUC (NMT val, турецкие отложенные)."""
    tk_fit = tk[ROLE[k].values == "fit"]; tk_iv = tk[ROLE[k].values == "iv"]
    X = [REC[i] for i in fit.id] + [REC[i] for i in tk_fit.id]
    y = np.r_[fit.y.to_numpy(), tk_fit.y.to_numpy()]; src = np.r_[np.zeros(len(fit)), np.ones(len(tk_fit))]
    w = np.zeros(len(y))
    for s_, share in [(0, 1 - SHARE_TK), (1, SHARE_TK)]:
        for c in (0, 1):
            m = (src == s_) & (y == c); w[m] = share * 0.5 / m.sum()
    w /= w.sum()
    iv12, y_iv = [TK12[i] for i in tk_iv.id], tk_iv.y.to_numpy()
    g = np.random.default_rng(k)
    torch.manual_seed(k)
    model = RVNet(POS).to(DEV)
    params = [p for p in model.parameters() if p.requires_grad]
    steps_ep = int(np.ceil(N_EP / BS))
    opt, sched = make_opt(model, MAX_EP * steps_ep)
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    best, best_state, best_ep, hist, n_bad, stop = -1, None, 0, [], 0, "макс. эпох"
    t0 = time.time()
    for ep in range(MAX_EP):
        model.train(); losses = []
        recs = g.choice(len(X), size=N_EP, replace=True, p=w)
        for b in range(0, N_EP, BS):
            j = recs[b:b + BS]
            xb = np.empty((len(j), 19, L), np.float16)
            for q, i in enumerate(j):
                s = int(g.integers(0, X[i].shape[1] - L + 1)); xb[q] = X[i][:, s:s + L]
            l, bad = step(model, opt, sched, scaler, augment(rv_input(xb).to(DEV), g), torch.tensor(y[j], device=DEV), params)
            n_bad += bad; losses.append(l)
        a_n = roc_auc_score(y_val, [np.median(p) for p in predict_windows(model, VAL12)])
        a_t = roc_auc_score(y_iv, [np.median(p) for p in predict_windows(model, iv12)])
        sel = (a_n + a_t) / 2
        hist.append((round(float(np.nanmean(losses)), 3), round(float(a_n), 3), round(float(a_t), 3)))
        print(f"  фолд {k} | эпоха {ep+1}: потери {np.nanmean(losses):.3f} | отложенные NMT {a_n:.3f}, Турция {a_t:.3f}, среднее {sel:.3f} | "
              f"NaN/inf {n_bad} | {time.time()-t0:.0f} с | осталось {left():.0f} мин", flush=True)
        if sel > best:
            best, best_ep, best_an, best_at = sel, ep + 1, a_n, a_t
            best_state = {kk: v.detach().cpu().clone() for kk, v in model.state_dict().items()}
        if ep + 1 - best_ep >= PATIENCE:
            stop = "плато"; break
        if time.time() + (time.time() - t0) / (ep + 1) > deadline:
            stop = "время"; break
    print(f"  фолд {k}: остановка — {stop}, лучшая эпоха {best_ep}, {(time.time()-t0)/60:.1f} мин", flush=True)
    model.load_state_dict(best_state)
    share = float(w[src == 1].sum())
    return model, dict(fold=k, seed=k, n_nmt_fit=len(fit), n_tk_fit=len(tk_fit), n_tk_iv=len(tk_iv), n_tk_test=int((ROLE[k] == "test").sum()),
                       turkey_share=round(share, 3), best_ep=best_ep, best_sel=round(float(best), 4), best_auc_nmt_val=round(float(best_an), 4),
                       best_auc_tk_iv=round(float(best_at), 4), hist=hist, n_bad=n_bad, stop=stop, n_ep=len(hist),
                       minutes=round((time.time() - t0) / 60, 1), min_per_ep=round((time.time() - t0) / 60 / len(hist), 2))

CFG_SAVE = dict(
    model="REVE v4-multi (Г1): дообучение на NMT train (fit) + 4/5 Турции, блокнот zakharsann/h1-reve-multisite",
    backbone="brain-bzh/reve-base", positions="brain-bzh/reve-positions", channels=EL, position_names=[NEW.get(e, e) for e in EL],
    head="Dropout(0.3) -> Linear(19*512, 256) -> GELU -> Dropout(0.3) -> Linear(256, 1); вход — выход основы (B, 19, T, 512), среднее по T, flatten",
    state_dict="ключи backbone.* (REVE) и head.*; fp32; класс RVNet в reve_multi_model.py (= reve_v4_model.py)",
    input="19 каналов EL, средний референт, 200 Гц, 0,1–75 Гц + 50 Гц + ФВЧ 0,3 Гц по записи; окно 10 с; rv_input: минус среднее канала, деление на общее СКО окна, обрезка ±15",
    output="логит; вероятность патологии/эпилепсии = sigmoid(логит)",
    train=dict(lr_backbone=LR_BACK, lr_head=LR_HEAD, weight_decay=0.05, batch=BS, windows_per_epoch=N_EP, max_epochs=MAX_EP, patience=PATIENCE,
               sampling=f"NMT {1-SHARE_TK:.0%} окон, Турция {SHARE_TK:.0%}; внутри источника классы поровну; каждое окно — новое случайное",
               epoch_choice="среднее ROC AUC: отложенные 15 % NMT train и отложенные 20 % турецкого обучения (12 окон, медиана)",
               augment="шум 0,1, зануление 1–2 каналов p=0,5, замена полушарий p=0,5, сдвиг до ±2 с"),
    record_score="среднее верхних 20 % вероятностей 60 окон 10 с (равномерно по записи)")

PRED_MIN, FOLDS, PRED = (0.5 if SMOKE else 3.0), [], {}
T_TRAIN = time.time()
for k in range(N_FOLDS):
    n_left = N_FOLDS - k
    budget = (left() - n_left * PRED_MIN - 3) / n_left
    model, inf = train_fold(k, time.time() + 60 * budget)
    ids_by_role = {r: ROLE[k][ROLE[k] == r].index.tolist() for r in ["fit", "iv", "test"]}
    torch.save({"state_dict": {kk: v.detach().cpu() for kk, v in model.state_dict().items()},
                "config": {**CFG_SAVE, "fold": k, "seed": k, "best_epoch": inf["best_ep"], "val_auc_nmt": inf["best_auc_nmt_val"],
                           "val_auc_turkey_iv": inf["best_auc_tk_iv"], "turkey_ids": ids_by_role}},
               f"{OUT}/reve_multi_fold{k}.pt")
    t0 = time.time()
    PRED[k] = {ds: predict_set(model, ds) for ds in GET}
    pm = (time.time() - t0) / 60; PRED_MIN = max(PRED_MIN, pm)
    FOLDS.append(inf)
    t20 = lambda P: np.array([np.sort(p)[::-1][:int(np.ceil(0.2 * len(p)))].mean() for p in P])
    te = (ROLE[k] == "test").values
    a = {ds: roc_auc_score(YS[ds], t20(PRED[k][ds])) for ds in ["nmt_eval", "epieeg60"]}
    a_te = roc_auc_score(YS["turkey"][te], t20(PRED[k]["turkey"])[te])
    print(f"фолд {k}: эпоха {inf['best_ep']} ({inf['stop']}, всего {inf['n_ep']}) | верхние 20 %: NMT eval {a['nmt_eval']:.3f}, "
          f"Турция (проверка фолда, n={te.sum()}) {a_te:.3f}, EpiEEG-60 {a['epieeg60']:.3f} | предсказания {pm:.1f} мин | осталось {left():.0f} мин", flush=True)
    del model
    if DEV == "cuda": torch.cuda.empty_cache()
print(f"всё обучение: {(time.time()-T_TRAIN)/60:.1f} мин, фолдов {len(PRED)}")

flash_attn not found, install it with `pip install flash_attn` if you want to use it
  фолд 0 | эпоха 1: потери 0.493 | отложенные NMT 0.876, Турция 1.000, среднее 0.938 | NaN/inf 0 | 212 с | осталось 132 мин
  фолд 0 | эпоха 2: потери 0.341 | отложенные NMT 0.894, Турция 1.000, среднее 0.947 | NaN/inf 0 | 432 с | осталось 128 мин
  фолд 0 | эпоха 3: потери 0.310 | отложенные NMT 0.879, Турция 1.000, среднее 0.940 | NaN/inf 0 | 651 с | осталось 124 мин
  фолд 0 | эпоха 4: потери 0.272 | отложенные NMT 0.896, Турция 1.000, среднее 0.948 | NaN/inf 0 | 869 с | осталось 121 мин
  фолд 0 | эпоха 5: потери 0.245 | отложенные NMT 0.896, Турция 1.000, среднее 0.948 | NaN/inf 0 | 1088 с | осталось 117 мин
  фолд 0 | эпоха 6: потери 0.232 | отложенные NMT 0.887, Турция 1.000, среднее 0.943 | NaN/inf 0 | 1306 с | осталось 113 мин
  фолд 0: остановка — плато, лучшая эпоха 4, 21.8 мин
фолд 0: эпоха 4 (плато, всего 6) | верхние 20 %: NMT eval 0.920, Турция (проверка фолда, n=25) 0.880, EpiEEG-60 0.6

## 3. Оценки по людям, ROC AUC, парный бутстреп «multi − v4»

In [ ]:
top20 = lambda p: np.sort(p)[::-1][:int(np.ceil(0.2 * len(p)))].mean()
AGGS = {"верхние 20 %": top20, "медиана": np.median}
KF = sorted(PRED)
FOLD_OF = tk.fold.to_numpy()
# SC[сборка][набор] — (фолды, люди)
SC = {a: {ds: np.array([[f(p) for p in PRED[k][ds]] for k in KF]) for ds in GET} for a, f in AGGS.items()}
ROLE_M = np.array([ROLE[k].values for k in KF])                       # (фолды, люди Турции)

def oof(a):
    """Турция: оценка каждого человека моделью фолда, где он был в проверке."""
    return SC[a]["turkey"][FOLD_OF, np.arange(len(tk))]
def iv_score(a):
    """Турция: среднее оценок моделей, где человек был отложенным (NaN, если ни разу)."""
    m = ROLE_M == "iv"; s = np.where(m, SC[a]["turkey"], np.nan)
    return np.where(m.any(0), np.nansum(s, 0) / np.maximum(m.sum(0), 1), np.nan)

# v4: оценки по людям (60 окон; ансамбль 3 запусков — среднее вероятностей)
v4pp = pd.read_csv(V4_PP); v4pp = v4pp[v4pp["окна"] == "60 окон"]
v4e = pd.read_csv(V4_EPI); v4e = v4e[v4e["окна"] == "60 окон"]
IDS = {"nmt_val": val.id, "nmt_eval": ev.id, "turkey": tk.id, "epieeg60": epi.person}
def v4_scores(ds, a):
    if ds == "epieeg60":
        return epi[["person"]].merge(v4e[v4e["сборка"] == a], on="person", how="left").reve4_ens_prob.to_numpy()
    d = v4pp[(v4pp.dataset == ds) & (v4pp["сборка"] == a)]
    return pd.DataFrame({"id": IDS[ds]}).merge(d, on="id", how="left").reve4_ens_prob.to_numpy()
IED = epi[["person"]].merge(v4e[v4e["сборка"] == "верхние 20 %"][["person", "n_ied_marks"]], on="person", how="left").n_ied_marks.to_numpy()
if not SMOKE:
    for ds in IDS:
        assert not np.isnan(v4_scores(ds, "верхние 20 %")).any(), ds

# таблица по людям
parts = []
for ds in GET:
    for a in AGGS:
        t = pd.DataFrame({"dataset": ds, "id": IDS[ds].to_numpy(), "y": YS[ds], "сборка": a})
        for j, k in enumerate(KF):
            t[f"multi_f{k}"] = SC[a][ds][j]
        if ds == "turkey":
            t["fold"] = FOLD_OF
            for k in KF: t[f"role_f{k}"] = ROLE[k].values
            t["multi_oof"] = oof(a); t["multi_iv"] = iv_score(a)
        t["multi_ens5"] = SC[a][ds].mean(0)
        t["multi_ens3"] = SC[a][ds][:3].mean(0)
        t["reve4_ens_prob"] = v4_scores(ds, a)
        if ds == "epieeg60": t["n_ied_marks"] = IED
        parts.append(t)
PP = pd.concat(parts, ignore_index=True)
PP.to_csv(f"{OUT}/reve_multi_per_person.csv", index=False)
def pp(ds, a="верхние 20 %"):
    return PP[(PP.dataset == ds) & (PP["сборка"] == a)].reset_index(drop=True)

def boot_ci(y, s, n=2000, seed=0):
    r = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y))
        if len(set(y[i])) == 2: v.append(roc_auc_score(y[i], s[i]))
    return np.percentile(v, [2.5, 97.5])
def fmt(y, s):
    lo, hi = boot_ci(y, s); return f"{roc_auc_score(y, s):.3f} ({lo:.2f}–{hi:.2f})"
def paired(y, a, b, n=2000, seed=0):
    """Парный бутстреп по людям: ROC AUC(a) − ROC AUC(b)."""
    r = np.random.default_rng(seed); d = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y))
        if len(set(y[i])) == 2: d.append(roc_auc_score(y[i], a[i]) - roc_auc_score(y[i], b[i]))
    d = np.array(d)
    return roc_auc_score(y, a) - roc_auc_score(y, b), np.percentile(d, 2.5), np.percentile(d, 97.5), float((d <= 0).mean())
sd = lambda v: v.std(ddof=1) if len(v) > 1 else 0.0

LAB = {"nmt_val": "NMT отложенные 15 %", "nmt_eval": "NMT eval", "turkey": "Турция", "epieeg60": "EpiEEG-60"}
srows = []
for a in AGGS:
    for ds in GET:
        d = pp(ds, a); y = d.y.to_numpy()
        runs = np.array([roc_auc_score(y, d[f"multi_f{k}"]) for k in KF])
        row = {"сборка": a, "набор": LAB[ds], "n": len(y), "n_класс1": int(y.sum())}
        if ds == "turkey":
            row["multi: основная оценка"] = "out-of-fold: " + fmt(y, d.multi_oof.to_numpy())
            row["multi: по фолдам (проверка фолда)"] = " / ".join(f"{roc_auc_score(y[d.fold == k], d.multi_oof[d.fold == k]):.2f}" for k in KF)
        else:
            row["multi: основная оценка"] = "ансамбль 5: " + fmt(y, d.multi_ens5.to_numpy())
            row["multi: ансамбль фолдов 0–2"] = fmt(y, d.multi_ens3.to_numpy())
            row["multi: модели фолдов"] = f"{runs.mean():.3f} ± {sd(runs):.3f} ({' / '.join(f'{x:.2f}' for x in runs)})"
        row["v4 (ансамбль 3)"] = fmt(y, d.reve4_ens_prob.to_numpy())
        srows.append(row)
SUM = pd.DataFrame(srows); SUM.to_csv(f"{OUT}/reve_multi_summary.csv", index=False)
pd.set_option("display.width", 250, "display.max_colwidth", 80)
print(SUM.to_string(index=False))

prows = []
for a in AGGS:
    for ds, col, note in [("epieeg60", "multi_ens5", "ансамбль 5 фолдов"), ("epieeg60", "multi_ens3", "ансамбль фолдов 0–2"),
                          ("nmt_eval", "multi_ens5", "ансамбль 5 фолдов"), ("nmt_eval", "multi_ens3", "ансамбль фолдов 0–2"),
                          ("turkey", "multi_oof", "out-of-fold; условия неравные: multi видела Турцию (других людей), v4 — нет")]:
        d = pp(ds, a); y = d.y.to_numpy()
        dd, lo, hi, p0 = paired(y, d[col].to_numpy(), d.reve4_ens_prob.to_numpy())
        prows.append({"сборка": a, "набор": LAB[ds], "multi": note, "n": len(y), "ROC AUC multi": round(roc_auc_score(y, d[col]), 3),
                      "ROC AUC v4": round(roc_auc_score(y, d.reve4_ens_prob), 3), "разность multi − v4": round(dd, 3),
                      "95% ДИ": f"{lo:+.3f}…{hi:+.3f}", "доля бутстрепа ≤ 0": round(p0, 3)})
PD = pd.DataFrame(prows); PD.to_csv(f"{OUT}/reve_multi_paired_differences.csv", index=False)
print(PD.to_string(index=False))

      сборка               набор   n  n_класс1         multi: основная оценка multi: ансамбль фолдов 0–2                             multi: модели фолдов   v4 (ансамбль 3) multi: по фолдам (проверка фолда)
верхние 20 % NMT отложенные 15 % 335        49  ансамбль 5: 0.914 (0.87–0.95)          0.914 (0.88–0.95) 0.902 ± 0.005 (0.91 / 0.90 / 0.89 / 0.90 / 0.91) 0.906 (0.86–0.94)                               NaN
верхние 20 %            NMT eval 185        90  ансамбль 5: 0.907 (0.87–0.94)          0.906 (0.87–0.94) 0.899 ± 0.024 (0.92 / 0.90 / 0.86 / 0.90 / 0.91) 0.913 (0.87–0.95)                               NaN
верхние 20 %              Турция 121        50 out-of-fold: 0.957 (0.92–0.99)                        NaN                                              NaN 0.923 (0.87–0.97)  0.88 / 1.00 / 0.99 / 0.96 / 0.97
верхние 20 %           EpiEEG-60  63        31  ансамбль 5: 0.712 (0.58–0.84)          0.718 (0.58–0.84) 0.699 ± 0.017 (0.70 / 0.70 / 0.70 / 0.67 / 0.72) 0.761 (0.63–0.88)     

## 4. Пороговые метрики: порог с отложенной части обучения

In [ ]:
def metr(y, pr):
    tp, tn = ((pr == 1) & (y == 1)).sum(), ((pr == 0) & (y == 0)).sum(); fp, fn = ((pr == 1) & (y == 0)).sum(), ((pr == 0) & (y == 1)).sum()
    rec = tp / max(tp + fn, 1); spec = tn / max(tn + fp, 1)
    return dict(acc=(tp + tn) / len(y), bacc=(rec + spec) / 2, prec=tp / (tp + fp) if tp + fp else np.nan, rec=rec, spec=spec)
def metr_ci(y, s, t, n=2000, seed=0):
    pr = (s >= t).astype(int); m0 = metr(y, pr); r = np.random.default_rng(seed); B = []
    for _ in range(n):
        i = r.integers(0, len(y), len(y)); B.append(metr(y[i], pr[i]))
    return {k: f"{m0[k]:.2f} ({np.nanpercentile([b[k] for b in B], 2.5):.2f}–{np.nanpercentile([b[k] for b in B], 97.5):.2f})" for k in m0}
def thr_youden(y, s):
    f, t, th = roc_curve(y, s); return float(th[np.argmax(t - f)])
def thr_sens(y, s, target=0.8):
    f, t, th = roc_curve(y, s); return float(th[np.where(t >= target)[0][0]])
RULES = {"Юден": thr_youden, "чувствительность 80 %": thr_sens}

trows, THR = [], {}
for a in AGGS:
    # порог ансамбля: NMT val (ансамбль 5) + турецкие отложенные (оценка модели фолда, где человек отложен)
    dv, dt = pp("nmt_val", a), pp("turkey", a)
    m_iv = dt.multi_iv.notna().to_numpy()
    yh = np.r_[dv.y.to_numpy(), dt.y.to_numpy()[m_iv]]; sh = np.r_[dv.multi_ens5.to_numpy(), dt.multi_iv.to_numpy()[m_iv]]
    for rn, rf in RULES.items():
        t = rf(yh, sh); THR[(a, rn)] = t
        for ds in ["nmt_eval", "epieeg60"]:
            d = pp(ds, a); y, s = d.y.to_numpy(), d.multi_ens5.to_numpy(); mm = metr_ci(y, s, t)
            trows.append({"модель": "REVE v4-multi, ансамбль 5", "сборка": a, "порог": rn, "значение порога": round(t, 3), "набор": LAB[ds], "n": len(y),
                          "примечание": f"порог: NMT val ({len(dv)}) + турецкие отложенные ({m_iv.sum()} чел.); перенесён без изменений",
                          "ROC AUC": fmt(y, s), "accuracy": mm["acc"], "сбаланс. accuracy": mm["bacc"], "precision": mm["prec"],
                          "recall (чувств.)": mm["rec"], "специфичность": mm["spec"]})
        # Турция out-of-fold: у каждого фолда свой порог по его отложенным (NMT val + турецкие iv, оценки модели фолда)
        tv = np.zeros(len(dt))
        for j, k in enumerate(KF):
            iv = (ROLE[k] == "iv").values
            tv[dt.fold.to_numpy() == k] = rf(np.r_[dv.y.to_numpy(), dt.y.to_numpy()[iv]], np.r_[dv[f"multi_f{k}"].to_numpy(), dt[f"multi_f{k}"].to_numpy()[iv]])
        y, s = dt.y.to_numpy(), dt.multi_oof.to_numpy(); mm = metr_ci(y, s, tv)
        trows.append({"модель": "REVE v4-multi, out-of-fold", "сборка": a, "порог": rn, "значение порога": " / ".join(f"{tv[dt.fold.to_numpy() == k][0]:.3f}" for k in KF),
                      "набор": "Турция", "n": len(y), "примечание": "порог свой у каждого фолда: NMT val + турецкие отложенные этого фолда (модель фолда)",
                      "ROC AUC": fmt(y, s), "accuracy": mm["acc"], "сбаланс. accuracy": mm["bacc"], "precision": mm["prec"],
                      "recall (чувств.)": mm["rec"], "специфичность": mm["spec"]})
TT = pd.DataFrame(trows); TT.to_csv(f"{OUT}/reve_multi_threshold_metrics.csv", index=False)
print(TT.drop(columns=["модель", "примечание"]).to_string(index=False))

      сборка                 порог                       значение порога     набор   n           ROC AUC         accuracy сбаланс. accuracy        precision recall (чувств.)    специфичность
верхние 20 %                  Юден                                 0.886  NMT eval 185 0.907 (0.87–0.94) 0.79 (0.74–0.85)  0.79 (0.73–0.85) 0.92 (0.85–0.98) 0.63 (0.53–0.73) 0.95 (0.90–0.99)
верхние 20 %                  Юден                                 0.886 EpiEEG-60  63 0.712 (0.58–0.84) 0.63 (0.52–0.75)  0.63 (0.52–0.74) 0.70 (0.50–0.90) 0.45 (0.29–0.63) 0.81 (0.67–0.94)
верхние 20 %                  Юден 0.816 / 0.956 / 0.830 / 0.942 / 0.792    Турция 121 0.957 (0.92–0.99) 0.88 (0.83–0.93)  0.89 (0.84–0.94) 0.81 (0.70–0.90) 0.94 (0.87–1.00) 0.85 (0.76–0.92)
верхние 20 % чувствительность 80 %                                 0.864  NMT eval 185 0.907 (0.87–0.94) 0.82 (0.76–0.87)  0.81 (0.76–0.87) 0.90 (0.82–0.96) 0.70 (0.60–0.79) 0.93 (0.87–0.97)
верхние 20 % чувствительность 80 %           

## 5. EpiEEG-60: калибровка порога на k = 10 / 20 людях и подгруппы по пометкам разрядов

In [ ]:
d = pp("epieeg60"); y = d.y.to_numpy()
MOD = {"REVE v4-multi (ансамбль 5)": d.multi_ens5.to_numpy(), "REVE v4 (ансамбль 3)": d.reve4_ens_prob.to_numpy()}
res = []
for name, s in MOD.items():
    for k in ([2, 3] if SMOKE else [10, 20]):
        rng = np.random.default_rng(0)
        for rep in range(500):
            idx = rng.permutation(len(d)); cal, test = idx[:k], idx[k:]
            if len(set(y[cal])) < 2 or len(set(y[test])) < 2: continue
            for rn, rf in RULES.items():
                th = rf(y[cal], s[cal])
                res.append(dict(модель=name, калибровка=f"{k} чел.", порог=rn, th=th, **metr(y[test], (s[test] >= th).astype(int))))
CR = pd.DataFrame(res); key = ["модель", "калибровка", "порог"]
g = CR.drop(columns="th").groupby(key, sort=False)
CAL = (g.mean().round(2).astype(str) + " (" + g.quantile(.025).round(2).astype(str) + "–" + g.quantile(.975).round(2).astype(str) + ")")
CAL = CAL.rename(columns={"acc": "accuracy", "bacc": "сбаланс. accuracy", "prec": "precision", "rec": "recall", "spec": "специфичность"}).reset_index()
gt = CR.groupby(key, sort=False).th
CAL.insert(3, "повторов", gt.size().values)
CAL.insert(4, "порог: медиана", gt.median().round(3).values)
# для сравнения — перенесённый порог (без калибровки на месте)
extra = []
for name, s, t in [("REVE v4-multi (ансамбль 5)", MOD["REVE v4-multi (ансамбль 5)"], THR[("верхние 20 %", "Юден")]), ("REVE v4 (ансамбль 3)", MOD["REVE v4 (ансамбль 3)"], 0.832)]:
    m0 = metr(y, (s >= t).astype(int))
    extra.append({"модель": name, "калибровка": "перенесён (Юден, отложенные обучения)", "порог": "Юден", "повторов": 1, "порог: медиана": round(t, 3),
                         "accuracy": f"{m0['acc']:.2f}", "сбаланс. accuracy": f"{m0['bacc']:.2f}", "precision": f"{m0['prec']:.2f}", "recall": f"{m0['rec']:.2f}", "специфичность": f"{m0['spec']:.2f}"})
CAL = pd.concat([CAL, pd.DataFrame(extra)], ignore_index=True)
CAL.to_csv(f"{OUT}/reve_multi_epieeg60_calibration.csv", index=False)
print(CAL.to_string(index=False))

SUB = []
for name, q in [("все", d), ("больные с пометками разрядов", d[(d.y == 0) | (d.n_ied_marks > 0)]), ("больные без пометок разрядов", d[(d.y == 0) | (d.n_ied_marks == 0)]),
                ("все больные; здоровые без пометок", d[(d.y == 1) | (d.n_ied_marks == 0)])]:
    yq = q.y.to_numpy(); a_, b_ = q.multi_ens5.to_numpy(), q.reve4_ens_prob.to_numpy()
    if len(set(yq)) < 2: continue
    dd, lo, hi, p0 = paired(yq, a_, b_)
    SUB.append(dict(подгруппа=name, n_норма=int((yq == 0).sum()), n_эпилепсия=int(yq.sum()), ROC_AUC_multi=fmt(yq, a_), ROC_AUC_v4=fmt(yq, b_),
                    разность=round(dd, 3), ДИ_разности=f"{lo:+.3f}…{hi:+.3f}",
                    recall_multi_при_пороге=round(float((a_[yq == 1] >= THR[("верхние 20 %", "Юден")]).mean()), 2),
                    recall_v4_при_пороге_0832=round(float((b_[yq == 1] >= 0.832).mean()), 2)))
SUB = pd.DataFrame(SUB); SUB.to_csv(f"{OUT}/reve_multi_epieeg60_subgroups.csv", index=False)
print(SUB.to_string(index=False))

                    модель                            калибровка                 порог  повторов  порог: медиана         accuracy сбаланс. accuracy        precision           recall    специфичность
REVE v4-multi (ансамбль 5)                               10 чел.                  Юден       499           0.826 0.64 (0.51–0.74)  0.64 (0.51–0.74) 0.67 (0.51–0.82) 0.61 (0.07–0.96)  0.68 (0.21–1.0)
REVE v4-multi (ансамбль 5)                               10 чел. чувствительность 80 %       499           0.611 0.63 (0.51–0.72)   0.63 (0.5–0.72) 0.62 (0.49–0.78)  0.74 (0.32–1.0)  0.52 (0.0–0.85)
REVE v4-multi (ансамбль 5)                               20 чел.                  Юден       500           0.826 0.66 (0.51–0.77)  0.67 (0.52–0.76)  0.69 (0.5–0.85)  0.63 (0.23–0.9)   0.7 (0.28–0.9)
REVE v4-multi (ансамбль 5)                               20 чел. чувствительность 80 %       500           0.611 0.63 (0.52–0.72)  0.63 (0.55–0.72)  0.61 (0.47–0.8)  0.77 (0.44–1.0)  0.5 (0.14–0.86)
     

## 6. Ход обучения, оценки окон, картинки

In [ ]:
RUNS = pd.DataFrame(FOLDS); RUNS.to_csv(f"{OUT}/reve_multi_runs.csv", index=False)
print(RUNS.to_string(index=False))
TKF.to_csv(f"{OUT}/reve_multi_turkey_folds.csv", index=False)
wparts = []
for ds in GET:
    for k in KF:
        P = PRED[k][ds]; n = [len(p) for p in P]
        wparts.append(pd.DataFrame({"dataset": ds, "id": np.repeat(IDS[ds].to_numpy(), n), "fold": k,
                                    "window": np.concatenate([np.arange(m) for m in n]), "p": np.concatenate(P).round(6)}))
pd.concat(wparts).to_csv(f"{OUT}/reve_multi_window_scores.csv.gz", index=False, compression="gzip")

plt.rcParams.update({"font.size": 10, "axes.titlesize": 10, "figure.dpi": 100})
def save_png(fig, name, limit=290_000):
    path = f"{FIG}/{name}"
    for dpi in [100, 85, 70, 60]:
        buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=dpi, bbox_inches="tight"); buf.seek(0)
        img = Image.open(buf)
        if buf.getbuffer().nbytes > limit:
            img = img.convert("RGB").quantize(256)
        img.save(path, optimize=True)
        if os.path.getsize(path) <= limit:
            break
    plt.close(fig)
C_V4, C_M = "#8a8a8a", "#2a78d6"
fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))
for j, (ds, col, nm) in enumerate([("epieeg60", "multi_ens5", "multi, ансамбль 5"), ("nmt_eval", "multi_ens5", "multi, ансамбль 5"), ("turkey", "multi_oof", "multi, out-of-fold")]):
    d = pp(ds); y = d.y.to_numpy()
    for s, c, lab in [(d.reve4_ens_prob.to_numpy(), C_V4, "v4 (только NMT)"), (d[col].to_numpy(), C_M, nm)]:
        f_, t_, _ = roc_curve(y, s); ax[j].plot(f_, t_, color=c, lw=2, label=f"{lab}: {roc_auc_score(y, s):.3f}")
    ax[j].plot([0, 1], [0, 1], ":", color="gray"); ax[j].legend(loc="lower right", fontsize=8)
    ax[j].set(xlabel="1 − специфичность", ylabel="чувствительность", title=f"{LAB[ds]} (n = {len(y)}), ROC AUC")
save_png(fig, "reve_multi_roc.png")

d = pp("epieeg60"); y = d.y.to_numpy()
fig, ax = plt.subplots(1, 2, figsize=(10, 4.2))
for c, col, nm in [(0, "#2a78d6", "норма"), (1, "#eb6834", "эпилепсия")]:
    m = y == c
    ax[0].scatter(d.reve4_ens_prob[m], d.multi_ens5[m], color=col, alpha=.75, label=nm)
mk = (y == 1) & (d.n_ied_marks.to_numpy() == 0)
ax[0].scatter(d.reve4_ens_prob[mk], d.multi_ens5[mk], facecolors="none", edgecolors="k", s=90, label="больные без пометок разрядов")
ax[0].plot([0, 1], [0, 1], ":", color="gray"); ax[0].legend(fontsize=8)
ax[0].set(xlabel="v4 (верхние 20 %, ансамбль 3)", ylabel="multi (верхние 20 %, ансамбль 5)", title="EpiEEG-60: оценки по людям")
for c, col, nm in [(0, "#2a78d6", "норма"), (1, "#eb6834", "эпилепсия")]:
    ax[1].scatter(np.full((y == c).sum(), c) + np.random.default_rng(c).uniform(-.15, .15, (y == c).sum()), d.multi_ens5[y == c], color=col, alpha=.75, label=nm)
ax[1].axhline(THR[("верхние 20 %", "Юден")], color="k", ls="--", lw=1, label=f"порог Юдена {THR[('верхние 20 %', 'Юден')]:.3f}")
ax[1].set_xticks([0, 1], ["норма", "эпилепсия"]); ax[1].set(ylabel="оценка multi", title="EpiEEG-60: multi и перенесённый порог"); ax[1].legend(fontsize=8)
save_png(fig, "reve_multi_epieeg60_scores.png")
print("файлы:", sorted(os.listdir(OUT)), sorted(os.listdir(FIG)))
print(f"\nвсё время блокнота: {(time.time()-T_NB)/60:.1f} мин")

 fold  seed  n_nmt_fit  n_tk_fit  n_tk_iv  n_tk_test  turkey_share  best_ep  best_sel  best_auc_nmt_val  best_auc_tk_iv                                                                                                                                                            hist  n_bad  stop  n_ep  minutes  min_per_ep
    0     0       1894        76       20         25           0.3        4    0.9482            0.8964          1.0000                                   [(0.493, 0.876, 1.0), (0.341, 0.894, 1.0), (0.31, 0.879, 1.0), (0.272, 0.896, 1.0), (0.245, 0.896, 1.0), (0.232, 0.887, 1.0)]      0 плато     6     21.8        3.63
    1     1       1894        77       20         24           0.3        4    0.9425            0.8849          1.0000                                [(0.517, 0.865, 1.0), (0.343, 0.867, 0.979), (0.294, 0.878, 1.0), (0.273, 0.885, 1.0), (0.242, 0.882, 1.0), (0.233, 0.874, 1.0)]      0 плато     6     21.9        3.65
    2     2       1894        77       2